<a href="https://colab.research.google.com/github/aliakseizvertouski/olist/blob/main/olist_geo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Анализ активности клиентов по geo

In [1]:
!git clone https://github.com/aliakseizvertouski/olist.git

Cloning into 'olist'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 94 (delta 50), reused 6 (delta 3), pack-reused 14 (from 1)
Receiving objects: 100% (94/94), 42.68 MiB | 18.25 MiB/s, done.
Resolving deltas: 100% (51/51), done.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
customers = pd.read_csv('/content/olist/olist_customers_dataset.csv')
geo = pd.read_csv('/content/olist/olist_geolocation_dataset.csv')
order_items = pd.read_csv('/content/olist/olist_order_items_dataset.csv')
order_payments = pd.read_csv('/content/olist/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('/content/olist/olist_order_reviews_dataset.csv')
orders = pd.read_csv('/content/olist/olist_orders_dataset.csv')
products = pd.read_csv('/content/olist/olist_products_dataset.csv')
sellers = pd.read_csv('/content/olist/olist_sellers_dataset.csv')
product_category_name_translation = pd.read_csv('/content/olist/product_category_name_translation.csv')

## Общее количество заказов по GEO

In [7]:
df_geo = customers.merge(orders, on='customer_id')

sales_stats = df_geo.groupby('customer_state')['order_id'].count().reset_index()
sales_stats = sales_stats.rename(columns={'order_id': 'sum_sales'}).sort_values(by='sum_sales', ascending=False)
sales_stats


,customer_state,sum_sales
25,SP,41746
18,RJ,12852
10,MG,11635
22,RS,5466
17,PR,5045
23,SC,3637
4,BA,3380
6,DF,2140
7,ES,2033
8,GO,2020


In [5]:
region_stats.to_csv('regional_sales.csv', index=False, sep=';')

## Популярность категорий по GEO

### Исходные данные

In [11]:
df_geo_cat = order_items[['order_id', 'product_id']].merge(
    orders[['order_id', 'customer_id']], on='order_id'
).merge(
    customers[['customer_id', 'customer_state']], on='customer_id'
).merge(
    products[['product_id', 'product_category_name']], on='product_id'
).merge(
    product_category_name_translation, on='product_category_name', how='left'
)

df_geo_cat = df_geo_cat.rename(columns={'product_category_name_english': 'category'})
df_geo_cat = df_geo_cat[['customer_state', 'category', 'product_id']]

df_geo_cat.head()

,customer_state,category,product_id
0,RJ,cool_stuff,4244733e06e7ecb4970a6e2683c13e61
1,SP,pet_shop,e5f2d52b802189ee658865ca93d83a8f
2,MG,furniture_decor,c777355d18b72b67abbeef9df44fd0fd
3,SP,perfumery,7634da152a4610f1595efa32f14722fc
4,SP,garden_tools,ac6c3623068f30de03045865e4e10089
...,...,...,...
112645,MA,housewares,4aa6014eceb682077f9dc4bffebc05b0
112646,PR,computers_accessories,32e07fd915822b0765e448c4dd74c828
112647,SP,sports_leisure,72a30483855e2eafc67aee5dc2560482
112648,SP,computers_accessories,9c422a519119dcad7575db5af1ba540e


### Самые популярные категории для каждого региона

In [12]:
cat_geo_stats = df_geo_cat.groupby(['customer_state', 'category'])['product_id'].count().reset_index()
cat_geo_stats = cat_geo_stats.rename(columns={'product_id': 'sales'})

cat_geo_stats = cat_geo_stats.sort_values(by=['customer_state', 'sales'], ascending=[True, False])

cat_geo_stats['rank'] = cat_geo_stats.groupby('customer_state').cumcount() + 1

top_3_per_state = cat_geo_stats[cat_geo_stats['rank'] <= 3]

top_3_per_state

,customer_state,category,sales,rank
12,AC,furniture_decor,12,1
6,AC,computers_accessories,9,2
22,AC,sports_leisure,9,3
50,AL,health_beauty,63,1
35,AL,computers_accessories,41,2
...,...,...,...,...
1291,SP,health_beauty,4204,2
1312,SP,sports_leisure,3667,3
1336,TO,health_beauty,36,1
1355,TO,watches_gifts,30,2


In [13]:
geo_cat_stats = df_geo_cat.groupby(['customer_state', 'category'])['product_id'].count().reset_index()
geo_cat_stats = geo_cat_stats.rename(columns={'product_id': 'sales_count'})

top_geo_cat = geo_cat_stats.sort_values(['customer_state', 'sales_count'], ascending=[True, False])
top_geo_cat = top_geo_cat.groupby('customer_state').head(3)                                                   # черт возьми как же это изящно

top_geo_cat

,customer_state,category,sales_count
12,AC,furniture_decor,12
6,AC,computers_accessories,9
22,AC,sports_leisure,9
50,AL,health_beauty,63
35,AL,computers_accessories,41
...,...,...,...
1291,SP,health_beauty,4204
1312,SP,sports_leisure,3667
1336,TO,health_beauty,36
1355,TO,watches_gifts,30


In [ ]:
top_geo_cat.to_csv('geo_categories.csv', index=False, sep=';')